# Modül 03: Kayıp Fonksiyonları ve Optimizasyon
## Deep Learning Path — Kapsamlı Jupyter Notebook

---
## 📋 İçindekiler
1. [Öğrenme Hedefleri](#1)
2. [Matematiksel Arka Plan](#2)
3. [Kayıp Fonksiyonları — NumPy](#3)
4. [Görsel Karşılaştırma](#4)
5. [Optimizasyon Algoritmaları](#5)
6. [Optimizer Yarışması](#6)
7. [Öğrenme Hızı Programları](#7)
8. [TensorFlow / Keras](#8)
9. [PyTorch](#9)
10. [Hiperparametre Deneyleri](#10)
11. [Özet ve Alıştırmalar](#11)


## 1. Öğrenme Hedefleri <a id='1'></a>
- ✅ MSE, MAE, Huber, BCE, CCE, KL kayıp fonksiyonlarını formüle etmek
- ✅ Hangi kayıp fonksiyonunun hangi probleme uygun olduğunu seçmek
- ✅ SGD, Momentum, RMSProp, Adam, AdaGrad algoritmalarını sıfırdan uygulamak
- ✅ Adam'ın neden Momentum + RMSProp kombinasyonu olduğunu açıklamak
- ✅ 5 farklı LR scheduler'ı karşılaştırmalı analiz etmek
- ✅ Üç framework'te kayıp ve optimizer kullanımını kodlamak


## 2. Matematiksel Arka Plan <a id='2'></a>

### Kayıp Fonksiyonları

| Kayıp | Formül | Problem Tipi |
|-------|--------|-------------|
| MSE | $\frac{1}{n}\sum(y-\hat{y})^2$ | Regresyon |
| MAE | $\frac{1}{n}\sum|y-\hat{y}|$ | Robust regresyon |
| Huber | $\begin{cases}\frac{1}{2}e^2 & |e|\leq\delta \\ \delta(|e|-\frac{\delta}{2}) & |e|>\delta\end{cases}$ | Aykırı değerli regresyon |
| BCE | $-\frac{1}{n}\sum[y\log\hat{y}+(1-y)\log(1-\hat{y})]$ | İkili sınıflandırma |
| CCE | $-\frac{1}{n}\sum_i\sum_k y_{ik}\log\hat{y}_{ik}$ | Çok sınıflı |

### Adam Optimizer

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\nabla W$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)(\nabla W)^2$$
$$W \leftarrow W - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

# ── Kayıp Fonksiyonları ────────────────────────────────────
def mse(yt,yp):     return np.mean((yt-yp)**2)
def mae(yt,yp):     return np.mean(np.abs(yt-yp))
def huber(yt,yp,d=1.):
    e=np.abs(yt-yp)
    return np.mean(np.where(e<=d, .5*e**2, d*(e-.5*d)))
def bce(yt,yp):
    eps=1e-15; yp=np.clip(yp,eps,1-eps)
    return -np.mean(yt*np.log(yp)+(1-yt)*np.log(1-yp))
def cce(yt,yp):
    eps=1e-15; yp=np.clip(yp,eps,1.)
    return -np.mean(np.sum(yt*np.log(yp),axis=1))
def kl_div(p,q):
    eps=1e-15; p=np.clip(p,eps,1.); q=np.clip(q,eps,1.)
    return np.sum(p*np.log(p/q))

# Test
y_true = np.array([1.,2.,3.,4.,5.])
y_good = np.array([1.1,1.9,3.2,3.8,5.1])
y_bad  = np.array([1.5,3.0,1.5,6.0,3.5])

print("Kayıp Fonksiyonu Karşılaştırması:")
print(f"{'Kayıp':<12} {'İyi tahmin':>12} {'Kötü tahmin':>13}")
print("-"*40)
for name,fn in [('MSE',mse),('MAE',mae),('Huber',lambda a,b:huber(a,b,1.))]:
    print(f"{name:<12} {fn(y_true,y_good):>12.4f} {fn(y_true,y_bad):>13.4f}")
print(f"\nBCE (güvenli): {bce(np.array([1.,0.,1.,0.]),np.array([.9,.1,.8,.2])):.4f}")
print(f"BCE (belirsiz): {bce(np.array([1.,0.,1.,0.]),np.array([.6,.4,.55,.45])):.4f}")


## 3. Görsel Karşılaştırma <a id='4'></a>

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,4))
errors = np.linspace(-3,3,300)

ax1 = axes[0]
ax1.plot(errors, errors**2, lw=2.5, label='MSE: e²', color='#1565C0')
ax1.plot(errors, np.abs(errors), lw=2.5, label='MAE: |e|', color='#E65100')
d=1.; hub=np.where(np.abs(errors)<=d,.5*errors**2,d*(np.abs(errors)-.5*d))
ax1.plot(errors, hub, lw=2.5, label='Huber (δ=1)', color='#00695C')
ax1.axvline(-1,color='gray',lw=1,ls=':',alpha=.7); ax1.axvline(1,color='gray',lw=1,ls=':',alpha=.7)
ax1.set_title('Regresyon Kayıp Fonksiyonları', fontweight='bold')
ax1.set_xlabel('Hata (y − ŷ)'); ax1.set_ylabel('Kayıp'); ax1.legend(); ax1.set_ylim(0,5)

p = np.linspace(.01,.99,300)
ax2 = axes[1]
ax2.plot(p,-np.log(p),    lw=2.5, color='#1565C0', label='BCE (y=1)')
ax2.plot(p,-np.log(1-p),  lw=2.5, color='#1565C0', ls='--', label='BCE (y=0)')
ax2.plot(p,(1-p)**2,      lw=2.5, color='#E65100', label='MSE (y=1)')
ax2.plot(p,p**2,          lw=2.5, color='#E65100', ls='--', label='MSE (y=0)')
ax2.set_title('BCE vs MSE — Sınıflandırma Kaybı', fontweight='bold')
ax2.set_xlabel('Tahmin ŷ'); ax2.set_ylabel('Kayıp')
ax2.legend(fontsize=8); ax2.set_ylim(0,4)

plt.tight_layout(); plt.show()
print("✓ BCE, yanlış tahmine log ölçeğinde büyük ceza verir → daha hızlı öğrenir")


## 4. Optimizasyon Algoritmaları <a id='5'></a>

### SGD → Adam Evrimi

| Algoritma | Güncelleme | Avantaj | Sorun |
|-----------|-----------|---------|-------|
| SGD | $W \leftarrow W - \eta\nabla W$ | Basit | Yavaş, tüm parametrelere aynı lr |
| Momentum | $v\leftarrow\beta v - \eta\nabla W$; $W\leftarrow W+v$ | Hızlı, yerel min'den çıkar | β seçimi |
| RMSProp | $W\leftarrow W-\frac{\eta}{\sqrt{s+\epsilon}}\nabla W$ | Uyarlanmış lr | lr birikimsiz |
| **Adam** | $W\leftarrow W-\eta\frac{\hat{m}}{\sqrt{\hat{v}}+\epsilon}$ | **Her ikisinin avantajı** | Bazen generaliz. |


In [ ]:
# Optimizerleri sıfırdan uygula
np.random.seed(0)
X = np.random.randn(100)
Y = 2.*X + 1. + .1*np.random.randn(100)  # Hedef: w=2, b=1

def grad(w,b):
    yp = w*X+b; e = yp-Y
    return (2/len(X))*np.dot(e,X), (2/len(X))*np.sum(e)

def loss(w,b): return float(np.mean((w*X+b-Y)**2))

class SGD:
    def __init__(self,lr=.1): self.lr=lr
    def step(self,w,b):
        dw,db=grad(w,b); return w-self.lr*dw, b-self.lr*db

class Momentum:
    def __init__(self,lr=.05,b=.9): self.lr,self.b,self.vw,self.vb=lr,b,0.,0.
    def step(self,w,b):
        dw,db=grad(w,b)
        self.vw=self.b*self.vw-self.lr*dw; self.vb=self.b*self.vb-self.lr*db
        return w+self.vw, b+self.vb

class Adam:
    def __init__(self,lr=.05,b1=.9,b2=.999,eps=1e-8):
        self.lr,self.b1,self.b2,self.eps=lr,b1,b2,eps
        self.mw=self.vw=self.mb=self.vb=0.; self.t=0
    def step(self,w,b):
        self.t+=1; dw,db=grad(w,b)
        self.mw=self.b1*self.mw+(1-self.b1)*dw; self.mb=self.b1*self.mb+(1-self.b1)*db
        self.vw=self.b2*self.vw+(1-self.b2)*dw**2; self.vb=self.b2*self.vb+(1-self.b2)*db**2
        mwh=self.mw/(1-self.b1**self.t); mbh=self.mb/(1-self.b1**self.t)
        vwh=self.vw/(1-self.b2**self.t); vbh=self.vb/(1-self.b2**self.t)
        return w-self.lr*mwh/(np.sqrt(vwh)+self.eps), b-self.lr*mbh/(np.sqrt(vbh)+self.eps)

opts = {'SGD':(SGD(),'#1565C0'), 'Momentum':(Momentum(),'#00695C'), 'Adam':(Adam(),'#AD1457')}
EPOCHS=150; results={}
for name,(opt,c) in opts.items():
    w,b,hist=0.,0.,[]
    for _ in range(EPOCHS):
        hist.append(loss(w,b)); w,b=opt.step(w,b)
    results[name]={'hist':hist,'w':w,'b':b,'c':c}
    print(f"{name:<10} Final loss={hist[-1]:.6f}  w={w:.4f}  b={b:.4f}")


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,4))

ax1 = axes[0]
for name,res in results.items():
    ax1.semilogy(res['hist'],label=name,color=res['c'],lw=2)
ax1.set_title('Optimizer Yakınsama — Loss (log)', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE Loss'); ax1.legend(); ax1.grid(True,alpha=.3)

ax2 = axes[1]
names = list(results.keys())
final_losses = [results[n]['hist'][-1] for n in names]
bars = ax2.bar(names, final_losses, color=[results[n]['c'] for n in names], edgecolor='white', lw=1.5)
ax2.set_title('Final Loss Karşılaştırması', fontweight='bold')
ax2.set_ylabel('MSE Loss')
for bar, val in zip(bars, final_losses):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0002,
             f'{val:.5f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout(); plt.show()


## 5. Öğrenme Hızı Programları <a id='7'></a>

In [ ]:
def step_decay(e, lr0=.1, drop=.5, n=50): return lr0*(drop**(e//n))
def exp_decay(e, lr0=.1, k=.01): return lr0*np.exp(-k*e)
def cosine(e, hi=.1, lo=.001, T=200): return lo+.5*(hi-lo)*(1+np.cos(np.pi*e/T))
def warmup_cos(e, hi=.1, wu=20, T=200):
    return hi*(e/wu) if e<wu else cosine(e-wu, hi, .001, T-wu)

ep = np.arange(200)
scheds = {
    'Sabit': ([.1]*200,'#9E9E9E'),
    'Adım Azalma': ([step_decay(e) for e in ep],'#1565C0'),
    'Üstel Azalma': ([exp_decay(e) for e in ep],'#E65100'),
    'Cosine': ([cosine(e) for e in ep],'#00695C'),
    'Warmup+Cosine': ([warmup_cos(e) for e in ep],'#AD1457'),
}

plt.figure(figsize=(11,4))
for name,(lrs,c) in scheds.items():
    plt.plot(lrs,label=name,color=c,lw=2)
plt.title('Öğrenme Hızı Programları Karşılaştırması', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Öğrenme Hızı'); plt.legend(); plt.grid(True,alpha=.3)
plt.tight_layout(); plt.show()
print("Warmup+Cosine: Transformer eğitiminin standart yaklaşımı")


## 6. Alıştırmalar <a id='11'></a>

**1.** Huber Loss'u δ=0.5 ve δ=2.0 ile test et. δ büyüdükçe MSE'ye mi MAE'ye mi yaklaşıyor?

**2.** `Nadam` optimizer'ını uygula: Momentum yerine Nesterov momentum + Adam kombinasyonu.

**3.** AdamW (weight decay düzeltmeli Adam) ile Adam'ı karşılaştır — fark ne?

**4.** Cosine annealing'de T parametresini 50, 100, 200 olarak değiştir ve eğitim üzerindeki etkisini gözlemle.

**5.** Learning rate finder uygula: lr'yi her adımda küçük artışlarla büyüt, loss en hızlı düşen noktayı bul.

---
**Sonraki Modül:** Modül 04 — Backpropagation (Zincir Kuralı, Hesaplama Grafı, Sıfırdan Impl.)
